# TP 1: LDA/QDA y optimización matemática de modelos

# Intro teórica

## Definición: Clasificador Bayesiano

Sean $k$ poblaciones, $x \in \mathbb{R}^p$ puede pertenecer a cualquiera $g \in \mathcal{G}$ de ellas. Bajo un esquema bayesiano, se define entonces $\pi_j \doteq P(G = j)$ la probabilidad *a priori* de que $X$ pertenezca a la clase *j*, y se **asume conocida** la distribución condicional de cada observable dado su clase $f_j \doteq f_{X|G=j}$.

De esta manera dicha probabilidad *a posteriori* resulta
$$
P(G|_{X=x} = j) = \frac{f_{X|G=j}(x) \cdot p_G(j)}{f_X(x)} \propto f_j(x) \cdot \pi_j
$$

La regla de decisión de Bayes es entonces
$$
H(x) \doteq \arg \max_{g \in \mathcal{G}} \{ P(G|_{X=x} = j) \} = \arg \max_{g \in \mathcal{G}} \{ f_j(x) \cdot \pi_j \}
$$

es decir, se predice a $x$ como perteneciente a la población $j$ cuya probabilidad a posteriori es máxima.

*Ojo, a no desesperar! $\pi_j$ no es otra cosa que una constante prefijada, y $f_j$ es, en su esencia, un campo escalar de $x$ a simplemente evaluar.*

## Distribución condicional

Para los clasificadores de discriminante cuadrático y lineal (QDA/LDA) se asume que $X|_{G=j} \sim \mathcal{N}_p(\mu_j, \Sigma_j)$, es decir, se asume que cada población sigue una distribución normal.

Por definición, se tiene entonces que para una clase $j$:
$$
f_j(x) = \frac{1}{(2 \pi)^\frac{p}{2} \cdot |\Sigma_j|^\frac{1}{2}} e^{- \frac{1}{2}(x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j)}
$$

Aplicando logaritmo (que al ser una función estrictamente creciente no afecta el cálculo de máximos/mínimos), queda algo mucho más práctico de trabajar:

$$
\log{f_j(x)} = -\frac{1}{2}\log |\Sigma_j| - \frac{1}{2} (x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j) + C
$$

Observar que en este caso $C=-\frac{p}{2} \log(2\pi)$, pero no se tiene en cuenta ya que al tener una constante aditiva en todas las clases, no afecta al cálculo del máximo.

## LDA

En el caso de LDA se hace una suposición extra, que es $X|_{G=j} \sim \mathcal{N}_p(\mu_j, \Sigma)$, es decir que las poblaciones no sólo siguen una distribución normal sino que son de igual matriz de covarianzas. Reemplazando arriba se obtiene entonces:

$$
\log{f_j(x)} =  -\frac{1}{2}\log |\Sigma| - \frac{1}{2} (x-\mu_j)^T \Sigma^{-1} (x- \mu_j) + C
$$

Ahora, como $-\frac{1}{2}\log |\Sigma|$ es común a todas las clases se puede incorporar a la constante aditiva y, distribuyendo y reagrupando términos sobre $(x-\mu_j)^T \Sigma^{-1} (x- \mu_j)$ se obtiene finalmente:

$$
\log{f_j(x)} =  \mu_j^T \Sigma^{-1} (x- \frac{1}{2} \mu_j) + C'
$$

## Entrenamiento/Ajuste

Obsérvese que para ambos modelos, ajustarlos a los datos implica estimar los parámetros $(\mu_j, \Sigma_j) \; \forall j = 1, \dots, k$ en el caso de QDA, y $(\mu_j, \Sigma)$ para LDA.

Estos parámetros se estiman por máxima verosimilitud, de manera que los estimadores resultan:

* $\hat{\mu}_j = \bar{x}_j$ el promedio de los $x$ de la clase *j*
* $\hat{\Sigma}_j = s^2_j$ la matriz de covarianzas estimada para cada clase *j*
* $\hat{\pi}_j = f_{R_j} = \frac{n_j}{n}$ la frecuencia relativa de la clase *j* en la muestra
* $\hat{\Sigma} = \frac{1}{n} \sum_{j=1}^k n_j \cdot s^2_j$ el promedio ponderado (por frecs. relativas) de las matrices de covarianzas de todas las clases. *Observar que se utiliza el estimador de MV y no el insesgado*

Es importante notar que si bien todos los $\mu, \Sigma$ deben ser estimados, la distribución *a priori* puede no inferirse de los datos sino asumirse previamente, utilizándose como entrada del modelo.

## Predicción

Para estos modelos, al igual que para cualquier clasificador Bayesiano del tipo antes visto, la estimación de la clase es por método *plug-in* sobre la regla de decisión $H(x)$, es decir devolver la clase que maximiza $\hat{f}_j(x) \cdot \hat{\pi}_j$, o lo que es lo mismo $\log\hat{f}_j(x) + \log\hat{\pi}_j$.

# Código provisto

Con el fin de no retrasar al alumno con cuestiones estructurales y/o secundarias al tema que se pretende tratar, se provee una base de código que **no es obligatoria de usar** pero se asume que resulta resulta beneficiosa.

In [1]:
import numpy as np
import pandas as pd
import numpy.linalg as LA
from scipy.linalg import cholesky, solve_triangular
from scipy.linalg.lapack import dtrtri

## Base code

In [2]:
class BaseBayesianClassifier:
  def __init__(self):
    pass

  def _estimate_a_priori(self, y):
    a_priori = np.bincount(y.flatten().astype(int)) / y.size
    # Q3: para que sirve bincount?
    return np.log(a_priori)

  def _fit_params(self, X, y):
    # estimate all needed parameters for given model
    raise NotImplementedError()

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    raise NotImplementedError()

  def fit(self, X, y, a_priori=None):
    # if it's needed, estimate a priori probabilities
    self.log_a_priori = self._estimate_a_priori(y) if a_priori is None else np.log(a_priori)

    # now that everything else is in place, estimate all needed parameters for given model
    self._fit_params(X, y)
    # Q4: por que el _fit_params va al final? no se puede mover a, por ejemplo, antes de la priori?

  def predict(self, X):
    # this is actually an individual prediction encased in a for-loop
    m_obs = X.shape[1]
    y_hat = np.empty(m_obs, dtype=int)

    for i in range(m_obs):
      y_hat[i] = self._predict_one(X[:,i].reshape(-1,1))

    # return prediction as a row vector (matching y)
    return y_hat.reshape(1,-1)

  def _predict_one(self, x):
    # calculate all log posteriori probabilities (actually, +C)
    log_posteriori = [ log_a_priori_i + self._predict_log_conditional(x, idx) for idx, log_a_priori_i
                  in enumerate(self.log_a_priori) ]

    # return the class that has maximum a posteriori probability
    return np.argmax(log_posteriori)

In [3]:
class QDA(BaseBayesianClassifier):

  def _fit_params(self, X, y):
    # estimate each covariance matrix
    self.inv_covs = [LA.inv(np.cov(X[:,y.flatten()==idx], bias=True))
                      for idx in range(len(self.log_a_priori))]
    # Q5: por que hace falta el flatten y no se puede directamente X[:,y==idx]?
    # Q6: por que se usa bias=True en vez del default bias=False?
    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]
    # Q7: que hace axis=1? por que no axis=0?

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    inv_cov = self.inv_covs[class_idx]
    unbiased_x =  x - self.means[class_idx]
    return 0.5*np.log(LA.det(inv_cov)) -0.5 * unbiased_x.T @ inv_cov @ unbiased_x

In [4]:
class TensorizedQDA(QDA):

    def _fit_params(self, X, y):
        # ask plain QDA to fit params
        super()._fit_params(X,y)

        # stack onto new dimension
        self.tensor_inv_cov = np.stack(self.inv_covs)
        self.tensor_means = np.stack(self.means)

    def _predict_log_conditionals(self,x):
        unbiased_x = x - self.tensor_means
        inner_prod = unbiased_x.transpose(0,2,1) @ self.tensor_inv_cov @ unbiased_x

        return 0.5*np.log(LA.det(self.tensor_inv_cov)) - 0.5 * inner_prod.flatten()

    def _predict_one(self, x):
        # return the class that has maximum a posteriori probability
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))

In [5]:
class QDA_Chol1(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.L_invs = [
        LA.inv(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True))
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = L_inv @ unbiased_x

    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()

In [6]:
class QDA_Chol2(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.Ls = [
        cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True)
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L = self.Ls[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = solve_triangular(L, unbiased_x, lower=True)

    return -np.log(L.diagonal().prod()) -0.5 * (y**2).sum()

In [7]:
class QDA_Chol3(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.L_invs = [
        dtrtri(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True), lower=1)[0]
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = L_inv @ unbiased_x

    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()

## Datasets

Observar que se proveen **4 datasets diferentes**, el código de ejemplo usa uno solo pero eso no significa que ustedes se limiten al mismo. También pueden usar otros datasets de su elección.

In [8]:
from sklearn.datasets import load_iris, fetch_openml, load_wine
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

def get_iris_dataset():
  data = load_iris()
  X_full = data.data
  y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
  return X_full, y_full

def get_penguins_dataset():
    # get data
    df, tgt = fetch_openml(name="penguins", return_X_y=True, as_frame=True, parser='auto')

    # drop non-numeric columns
    df.drop(columns=["island","sex"], inplace=True)

    # drop rows with missing values
    mask = df.isna().sum(axis=1) == 0
    df = df[mask]
    tgt = tgt[mask]

    return df.values, tgt.to_numpy().reshape(-1,1)

def get_wine_dataset():
    # get data
    data = load_wine()
    X_full = data.data
    y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
    return X_full, y_full

def get_letters_dataset():
    # get data
    letter = fetch_openml('letter', version=1, as_frame=False)
    return letter.data, letter.target.reshape(-1,1)

def label_encode(y_full):
    return LabelEncoder().fit_transform(y_full.flatten()).reshape(y_full.shape)

def split_transpose(X, y, test_size, random_state):
    # X_train, X_test, y_train, y_test but all transposed
    return [elem.T for elem in train_test_split(X, y, test_size=test_size, random_state=random_state)]

## Benchmarking

Nota: esta clase fue creada bastante rápido y no pretende ser una plataforma súper confiable sobre la que basarse, sino más bien una herramienta simple con la que poder medir varios runs y agregar la información.

En forma rápida, `warmup` es la cantidad de runs para warmup, `mem_runs` es la cantidad de runs en las que se mide el pico de uso de RAM y `n_runs` es la cantidad de runs en las que se miden tiempos.

La razón por la que se separan es que medir memoria hace ~2.5x más lento cada run, pero al mismo tiempo se estabiliza mucho más rápido.

**Importante:** tener en cuenta que los modelos que predicen en batch (usan `predict` directamente) deberían consumir, como mínimo, $n$ veces la memoria de los que predicen por observación.

In [9]:
import time
from tqdm.notebook import tqdm
from numpy.random import RandomState
import tracemalloc

RNG_SEED = 6553

class Benchmark:
    def __init__(self, X, y, n_runs=1000, warmup=100, mem_runs=100, test_sz=0.3, rng_seed=RNG_SEED, same_splits=True):
        self.X = X
        self.y = y
        self.n = n_runs
        self.warmup = warmup
        self.mem_runs = mem_runs
        self.test_sz = test_sz
        self.det = same_splits
        if self.det:
            self.rng_seed = rng_seed
        else:
            self.rng = RandomState(rng_seed)

        self.data = dict()

        print("Benching params:")
        print("Total runs:",self.warmup+self.mem_runs+self.n)
        print("Warmup runs:",self.warmup)
        print("Peak Memory usage runs:", self.mem_runs)
        print("Running time runs:", self.n)
        approx_test_sz = int(self.y.size * self.test_sz)
        print("Train size rows (approx):",self.y.size - approx_test_sz)
        print("Test size rows (approx):",approx_test_sz)
        print("Test size fraction:",self.test_sz)

    def bench(self, model_class, **kwargs):
        name = model_class.__name__
        time_data = np.empty((self.n, 3), dtype=float)  # train_time, test_time, accuracy
        mem_data = np.empty((self.mem_runs, 2), dtype=float)  # train_peak_mem, test_peak_mem
        rng = RandomState(self.rng_seed) if self.det else self.rng


        for i in range(self.warmup):
            # Instantiate model with error check for unsupported parameters
            model = model_class(**kwargs)

            # Generate current train-test split
            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )
            # Run training and prediction (timing or memory measurement not recorded)
            model.fit(X_train, y_train)
            model.predict(X_test)

        for i in tqdm(range(self.mem_runs), total=self.mem_runs, desc=f"{name} (MEM)"):

            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            tracemalloc.start()

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()

            _, train_peak = tracemalloc.get_traced_memory()
            tracemalloc.reset_peak()

            model.predict(X_test)
            t3 = time.perf_counter()
            _, test_peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            mem_data[i,] = (
                train_peak / (1024 * 1024),
                test_peak / (1024 * 1024)
            )

        for i in tqdm(range(self.n), total=self.n, desc=f"{name} (TIME)"):
            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()
            preds = model.predict(X_test)
            t3 = time.perf_counter()

            time_data[i,] = (
                (t2 - t1) * 1000,
                (t3 - t2) * 1000,
                (y_test.flatten() == preds.flatten()).mean()
            )

        self.data[name] = (time_data, mem_data)

    def summary(self, baseline=None):
        aux = []
        for name, (time_data, mem_data) in self.data.items():
            result = {
                'model': name,
                'train_median_ms': np.median(time_data[:, 0]),
                'train_std_ms': time_data[:, 0].std(),
                'test_median_ms': np.median(time_data[:, 1]),
                'test_std_ms': time_data[:, 1].std(),
                'mean_accuracy': time_data[:, 2].mean(),
                'train_mem_median_mb': np.median(mem_data[:, 0]),
                'train_mem_std_mb': mem_data[:, 0].std(),
                'test_mem_median_mb': np.median(mem_data[:, 1]),
                'test_mem_std_mb': mem_data[:, 1].std()
            }
            aux.append(result)
        df = pd.DataFrame(aux).set_index('model')

        if baseline is not None and baseline in self.data:
            df['train_speedup'] = df.loc[baseline, 'train_median_ms'] / df['train_median_ms']
            df['test_speedup'] = df.loc[baseline, 'test_median_ms'] / df['test_median_ms']
            df['train_mem_reduction'] = df.loc[baseline, 'train_mem_median_mb'] / df['train_mem_median_mb']
            df['test_mem_reduction'] = df.loc[baseline, 'test_mem_median_mb'] / df['test_mem_median_mb']
        return df

## Ejemplo

In [10]:
# levantamos el dataset Wine, que tiene 13 features y 178 observaciones en total
X_full, y_full = get_wine_dataset()

X_full.shape, y_full.shape

((178, 13), (178, 1))

In [11]:
# encodeamos a número las clases
y_full_encoded = label_encode(y_full)

y_full[:5], y_full_encoded[:5]

(array([['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0']], dtype='<U7'),
 array([[0],
        [0],
        [0],
        [0],
        [0]]))

In [12]:
# generamos el benchmark
# observar que son valores muy bajos de runs para que corra rápido ahora
b = Benchmark(
    X_full, y_full_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


In [13]:
# bencheamos un par
to_bench = [QDA]

for model in to_bench:
    b.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [14]:
# como es una clase, podemos seguir bencheando más después
b.bench(TensorizedQDA)

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [15]:
# hacemos un summary
b.summary()

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb
model,,,,,,,,,
QDA,0.065604,0.049829,0.666750,0.082756,0.982407,0.018471,0.000615,0.007697,0.000419
TensorizedQDA,0.067604,0.048803,0.315312,0.046998,0.982593,0.018471,0.000653,0.012123,0.000210


In [16]:
# son muchos datos! nos quedamos con un par nomás
summ = b.summary()

# como es un pandas DataFrame, subseteamos columnas fácil
summ[['train_median_ms', 'test_median_ms','mean_accuracy']]

,train_median_ms,test_median_ms,mean_accuracy
model,,,
QDA,0.065604,0.666750,0.982407
TensorizedQDA,0.067604,0.315312,0.982593


In [17]:
# podemos setear un baseline para que fabrique columnas de comparación
summ = b.summary(baseline='QDA')

summ

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,0.065604,0.049829,0.666750,0.082756,0.982407,0.018471,0.000615,0.007697,0.000419,1.000000,1.000000,1.0,1.000000
TensorizedQDA,0.067604,0.048803,0.315312,0.046998,0.982593,0.018471,0.000653,0.012123,0.000210,0.970409,2.114569,1.0,0.634912


In [18]:
summ[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.065604,0.666750,0.982407,1.000000,1.000000,1.0,1.000000
TensorizedQDA,0.067604,0.315312,0.982593,0.970409,2.114569,1.0,0.634912


# Consigna QDA

**Notación**: en general notamos

* $k$ la cantidad de clases
* $n$ la cantidad de observaciones
* $p$ la cantidad de features/variables/predictores

**Sugerencia:** combinaciones adecuadas de `transpose`, `stack`, `reshape` y, ocasionalmente, `flatten` y `diagonal` suele ser más que suficiente. Se recomienda *fuertemente* explorar la dimensionalidad de cada elemento antes de implementar las clases.

## Tensorización

En esta sección nos vamos a ocupar de hacer que el modelo sea más rápido para generar predicciones, observando que incurre en un doble `for` dado que predice en forma individual un escalar para cada observación, para cada clase. Paralelizar ambos vía tensorización suena como una gran vía de mejora de tiempos.

### 1) Diferencias entre `QDA`y `TensorizedQDA`

1. ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?
2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.

### 2) Optimización

Debido a la forma cuadrática de QDA, no se puede predecir para $n$ observaciones en una sola pasada (utilizar $X \in \mathbb{R}^{p \times n}$ en vez de $x \in \mathbb{R}^p$) sin pasar por una matriz de $n \times n$ en donde se computan todas las interacciones entre observaciones. Se puede acceder al resultado recuperando sólo la diagonal de dicha matriz, pero resulta ineficiente en tiempo y (especialmente) en memoria. Aún así, es *posible* que el modelo funcione más rápido.

3. Implementar el modelo `FasterQDA` (se recomienda heredarlo de `TensorizedQDA`) de manera de eliminar el ciclo for en el método predict.
4. Mostrar dónde aparece la mencionada matriz de $n \times n$, donde $n$ es la cantidad de observaciones a predecir.
5. Demostrar que
$$
diag(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis=1)
$$ es decir, que se puede "esquivar" la matriz de $n \times n$ usando matrices de $n \times p$. También se puede usar, de forma equivalente,
$$
np.sum(A^T \odot B, axis=0).T
$$
queda a preferencia del alumno cuál usar.
6. Utilizar la propiedad antes demostrada para reimplementar la predicción del modelo `FasterQDA` de forma eficiente en un nuevo modelo `EfficientQDA`.
7. Comparar la performance de las 4 variantes de QDA implementadas hasta ahora (no Cholesky) ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

## Cholesky

Hasta ahora todos los esfuerzos fueron enfocados en realizar una predicción más rápida. Los tiempos de entrenamiento (teóricos al menos) siguen siendo los mismos o hasta (minúsculamente) peores, dado que todas las mejoras siguen llamando al método `_fit_params` original de `QDA`.

La descomposición/factorización de [Cholesky](https://en.wikipedia.org/wiki/Cholesky_decomposition#Statement) permite factorizar una matriz definida positiva $A = LL^T$ donde $L$ es una matriz triangular inferior. En particular, si bien se asume que $p \ll n$, invertir la matriz de covarianzas $\Sigma$ para cada clase impone un cuello de botella que podría alivianarse. Teniendo en cuenta que las matrices de covarianza son simétricas y salvo degeneración, definidas positivas, Cholesky como mínimo debería permitir invertir la matriz más rápido.

*Nota: observar que calcular* $A^{-1}b$ *equivale a resolver el sistema* $Ax=b$.

### 3) Diferencias entre implementaciones de `QDA_Chol`

8. Si una matriz $A$ tiene fact. de Cholesky $A=LL^T$, expresar $A^{-1}$ en términos de $L$. ¿Cómo podría esto ser útil en la forma cuadrática de QDA?
7. Explicar las diferencias entre `QDA_Chol1`y `QDA` y cómo `QDA_Chol1` llega, paso a paso, hasta las predicciones.
8. ¿Cuáles son las diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`?
9. Comparar la performance de las 7 variantes de QDA implementadas hasta ahora ¿Qué se observa?¿Hay alguna de las implementaciones de `QDA_Chol` que sea claramente mejor que las demás?¿Alguna que sea peor?

### 4) Optimización

12. Implementar el modelo `TensorizedChol` paralelizando sobre clases/observaciones según corresponda. Se recomienda heredarlo de alguna de las implementaciones de `QDA_Chol`, aunque la elección de cuál de ellas queda a cargo del alumno según lo observado en los benchmarks de puntos anteriores.
13. Implementar el modelo `EfficientChol` combinando los insights de `EfficientQDA` y `TensorizedChol`. Si se desea, se puede implementar `FasterChol` como ayuda, pero no se contempla para el punto.
13. Comparar la performance de las 9 variantes de QDA implementadas ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

## Importante:

Las métricas que se observan al realizar benchmarking son muy dependientes del código que se ejecuta, y por tanto de las versiones de las librerías utilizadas. Una forma de unificar esto es utilizando un gestor de versiones y paquetes como _uv_ o _Poetry_, otra es simplemente usando una misma VM como la que provee Colab.

**Cada equipo debe informar las versiones de Python, NumPy y SciPy con que fueron obtenidos los resultados. En caso de que sean múltiples, agregar todos los casos**. La siguiente celda provee una ayuda para hacerlo desde un notebook, aunque como es una secuencia de comandos también sirve para consola.

In [19]:
%%bash
python --version
pip freeze | grep -E "scipy|numpy"

Python 3.12.13


numpy==2.3.1
scipy==1.16.0


**Comentario:** yo utilicé los siguientes parámetros para mi run de prueba. Esto NO significa que ustedes tengan que usar los mismos, tampoco el mismo dataset. Se agregó al notebook simplemente porque fue una pregunta común en cohortes anteriores.

In [20]:
# dataset de letters
X_letter, y_letter = get_letters_dataset()

# encoding de labels
y_letter_encoded = label_encode(y_letter.reshape(-1,1))

# instanciacion del benchmark
b = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


---
# ✅ RESOLUCIÓN DEL TP

**Alumno:** Gustavo Varela · **Materia:** Análisis Matemático para IA (AMIA), CEIA-FIUBA

Todo lo de arriba es el material provisto por la cátedra (teoría + código base + consigna).
De acá en adelante va la resolución: las clases base (`QDA`, `TensorizedQDA`, `QDA_Chol1/2/3`,
`Benchmark`, datasets) ya están definidas en las celdas anteriores; acá defino **inline** las
4 clases nuevas y respondo las preguntas.

> En cada respuesta: primero **la idea** (el *por qué*), después **el detalle** (el *cómo*).
> Notación de la consigna: $k$ = clases, $n$ = observaciones, $p$ = features; $X\in\mathbb{R}^{p\times n}$.

## 0. Reproducibilidad — versiones

In [21]:
import sys, numpy, scipy, sklearn
print("Python :", sys.version.split()[0])
print("NumPy  :", numpy.__version__)
print("SciPy  :", scipy.__version__)
print("sklearn:", sklearn.__version__)

Python : 3.12.13
NumPy  : 2.3.1
SciPy  : 1.16.0
sklearn: 1.7.0


## 1. Preguntas sobre el código base (Q3–Q7)

> Cada respuesta va en tres capas: **Concepto** (la intuición) → **En lo técnico** (la mecánica) → **Respuesta** (el cierre directo).

---

> **Q3 · ¿Para qué sirve `bincount`?**

**Concepto.** Un clasificador bayesiano necesita saber, antes de mirar las features, qué tan probable es cada clase: las a priori $\pi_j$. Lo natural es estimarlas contando qué fracción de la muestra cae en cada clase.

**En lo técnico.** `np.bincount(y)` recorre las etiquetas enteras y devuelve $[n_0,\dots,n_{k-1}]$. Dividido por `y.size` da $\hat\pi_j=n_j/n$; el `np.log` lleva todo a escala logarítmica (sumar en vez de multiplicar).

**Respuesta.** `bincount` cuenta las frecuencias absolutas por clase; normalizadas son el estimador de máxima verosimilitud de las probabilidades a priori $\pi_j$.

---

> **Q4 · ¿Por qué `_fit_params` va al final, después de las a priori?**

**Concepto.** Hay un orden de dependencia: para estimar los parámetros de cada clase, el modelo primero tiene que saber cuántas clases hay — y eso lo fija el cálculo de las a priori. Además, las a priori son lo único que el usuario puede imponer a mano en lugar de estimar.

**En lo técnico.** `_fit_params` itera con `len(self.log_a_priori)`; ese atributo se crea en `_estimate_a_priori`, que corre antes. Moverlo antes ⇒ `self.log_a_priori` no existe ⇒ `AttributeError`.

**Respuesta.** No se puede mover: `_fit_params` depende de `self.log_a_priori` (el número de clases), que recién existe tras estimar las a priori. Es una dependencia real, no un orden arbitrario.

---

> **Q5 · ¿Por qué el `flatten` y no directamente `X[:, y==idx]`?**

**Concepto.** Estamos separando las columnas de $X$ (las observaciones) por clase; esa selección tiene que ser una marca plana, una por observación.

**En lo técnico.** `y` es `(1, n)` (2-D), así que `y==idx` es una máscara booleana `(1, n)`. Indexar el eje de columnas de `X` (`(p, n)`) requiere una máscara 1-D de largo $n$; una 2-D intenta indexar más ejes de los que hay.

**Respuesta.** Sin `flatten` la máscara es 2-D y el indexado falla; `y.flatten()` la lleva a `(n,)` para que `X[:, y.flatten()==idx]` seleccione las columnas de la clase.

---

> **Q6 · ¿Por qué `bias=True` en vez del default `bias=False`?**

**Concepto.** Hay dos estimadores de covarianza: el insesgado (divide por $n-1$) y el de máxima verosimilitud (divide por $n$). QDA se deriva entero por máxima verosimilitud, así que la covarianza debe estimarse con el mismo criterio para ser coherente.

**En lo técnico.** `np.cov` con `bias=False` normaliza por $n-1$; con `bias=True`, por $n$.

**Respuesta.** Se usa `bias=True` porque el estimador MV de $\Sigma$ divide por $n$ — el que pide la teoría del modelo, no el insesgado.

---

> **Q7 · ¿Qué hace `axis=1`? ¿Por qué no `axis=0`?**

**Concepto.** La media de una clase es su centro de gravedad: un promedio por feature, calculado a lo largo de las observaciones de esa clase.

**En lo técnico.** $X$ es `(p, n)`: features en filas, observaciones en columnas. `axis=1` promedia a lo largo de las columnas (observaciones) → `(p, 1)`; `axis=0` promediaría sobre las features. `keepdims=True` conserva `(p,1)` para el broadcasting contra $x$.

**Respuesta.** `axis=1` promedia sobre las observaciones y da el vector de medias `(p,1)`; `axis=0` mezclaría features distintas y no tiene sentido.

## 2. Tensorización

> **Concepto de fondo.** El modelo no cambia: siempre se elige la clase que maximiza log-priori + log-verosimilitud. Cambia *cómo* se computa. `QDA` usa dos `for` anidados en Python (observaciones × clases); tensorizar es reemplazar esos bucles por álgebra matricial que NumPy ejecuta en C — mismo resultado, sin el overhead del intérprete.

---

> **P1 · ¿Sobre qué paraleliza `TensorizedQDA`? ¿Las $k$ clases, las $n$ observaciones, o ambas?**

**Concepto.** Da el primer paso, no el completo: paraleliza sobre las clases, pero sigue prediciendo de a una observación.

**En lo técnico.** Apila las matrices por clase en tensores y calcula las $k$ log-condicionales de una observación en una operación batcheada; el `for i in range(m_obs)` de `predict` queda intacto.

**Respuesta.** Paraleliza **solo sobre las $k$ clases**, no sobre las $n$ observaciones (eso lo hará `FasterQDA`).

---

> **P2 · Shapes de `tensor_inv_cov` y `tensor_means`, y por qué predice lo mismo que `QDA`.**

**Concepto.** Apilar las matrices/medias en una dimensión extra deja que una sola multiplicación batcheada calcule, en paralelo, la misma forma cuadrática que `QDA` hacía clase por clase.

**En lo técnico.** `tensor_inv_cov`: `(k, p, p)`; `tensor_means`: `(k, p, 1)`. Para $x$ de `(p,1)`:

| paso | operación | shape |
|---|---|---|
| centrado | `x - tensor_means` | `(k, p, 1)` |
| traspuesta | `.transpose(0,2,1)` | `(k, 1, p)` |
| $\cdot\,\Sigma^{-1}$ | `@ tensor_inv_cov` | `(k, 1, p)` |
| $\cdot\,(x-\mu)$ | `@ unbiased` | `(k, 1, 1)` |
| `flatten` | | `(k,)` |

**Respuesta.** Ese `(k,)` es, por clase, $(x-\mu_j)^T\Sigma_j^{-1}(x-\mu_j)$ —idéntico al loop de `QDA`—; sumando $\tfrac12\log|\Sigma_j^{-1}|$ y la log-priori se llega al mismo `argmax`. Cambia la mecánica, no el número.

> **P3 · Implementar `FasterQDA` eliminando el `for` de `predict`.**

**Concepto.** A `TensorizedQDA` le falta sacar el otro bucle, el de las $n$ observaciones. `FasterQDA` procesa toda la matriz $X\in\mathbb{R}^{p\times n}$ de una pasada, paralelizando clases y observaciones.

**En lo técnico.** Hereda de `TensorizedQDA` y reescribe `predict` operando sobre `unbiased` de shape `(k, p, n)`. (Código abajo.)

**Respuesta.** `FasterQDA` reemplaza el `for` por una operación tensorial sobre las $n$ observaciones — pero, como muestra P4, paga un costo de memoria.

---

> **P4 · Mostrar dónde aparece la matriz de $n\times n$.**

**Concepto.** Hacer "todo de una" tiene un costo escondido: el álgebra obliga a computar *todos* los productos cruzados entre observaciones, no solo los $n$ que importan.

**En lo técnico.** `unbiased.transpose(0,2,1) @ tensor_inv_cov @ unbiased` con `unbiased` `(k, p, n)` da `(k, n, n)`. La diagonal de cada bloque son las $n$ formas cuadráticas útiles; fuera de la diagonal, las interacciones $(x_i-\mu)^T\Sigma^{-1}(x_j-\mu)$ que se descartan.

**Respuesta.** La matriz $n\times n$ aparece en ese triple producto: es `(k, n, n)`, de la que solo se usa la diagonal. Es $O(n^2)$ en cómputo y memoria — innecesario.

In [22]:
class FasterQDA(TensorizedQDA):
    """P3) Elimina el ciclo for de predict: paraleliza sobre clases Y observaciones.
    P4) El precio es la matriz n x n: unbiased.transpose(0,2,1) @ inv_cov @ unbiased
    con unbiased (k, p, n) da (k, n, n); solo se usa su diagonal."""

    def predict(self, X):
        unbiased = X[np.newaxis, :, :] - self.tensor_means            # (k, p, n)
        prod = unbiased.transpose(0, 2, 1) @ self.tensor_inv_cov @ unbiased  # (k, n, n)
        quad = np.diagonal(prod, axis1=1, axis2=2)                    # (k, n)
        log_dets = 0.5 * np.log(LA.det(self.tensor_inv_cov))[:, np.newaxis]
        log_conditionals = log_dets - 0.5 * quad
        log_posteriori = self.log_a_priori[:, np.newaxis] + log_conditionals
        return np.argmax(log_posteriori, axis=0).reshape(1, -1)

> **P5 · Demostrar que $\operatorname{diag}(A\cdot B)=\sum_{\text{cols}}A\odot B^T=\texttt{np.sum}(A\odot B^T,\ \texttt{axis=1})$.**

**Concepto.** La matriz $n\times n$ es un desperdicio: calculamos $n^2$ números para usar $n$. Cada elemento de la diagonal de $AB$ es un producto punto (fila de $A$ × columna de $B$), y eso se obtiene multiplicando elemento a elemento y sumando — sin construir $AB$.

**En lo técnico (demostración).** Con $A\in\mathbb{R}^{n\times p}$, $B\in\mathbb{R}^{p\times n}$:
$$(AB)_{ii}=\sum_{j=1}^{p}A_{ij}B_{ji}.$$
Como $(B^T)_{ij}=B_{ji}$, el producto de Hadamard da $(A\odot B^T)_{ij}=A_{ij}B_{ji}$. Sumando sobre $j$ (las columnas, `axis=1`):
$$\sum_{j=1}^{p}(A\odot B^T)_{ij}=\sum_{j=1}^{p}A_{ij}B_{ji}=(AB)_{ii}.\qquad\blacksquare$$

**Respuesta.** La diagonal de $AB$ se obtiene con matrices $n\times p$ (`np.sum(A⊙B^T, axis=1)`), sin construir nunca la $n\times n$. Equivale a `np.sum(A^T⊙B, axis=0).T`.

---

> **P6 · Reimplementar la predicción eficiente en `EfficientQDA`.**

**Concepto.** Aplicamos P5 a la forma cuadrática: con $A=(x-\mu)^T$ y $B=\Sigma^{-1}(x-\mu)$, la diagonal buscada es "centrar, aplicar $\Sigma^{-1}$, y sumar el producto elemento a elemento sobre las features".

**En lo técnico.** $(x-\mu)^T\Sigma^{-1}(x-\mu)=\sum_p\big[\texttt{unbiased}\odot(\Sigma^{-1}\texttt{unbiased})\big]$, todo en `(k, p, n)`. (Código abajo.)

**Respuesta.** `EfficientQDA` da el mismo resultado que `FasterQDA` pero con memoria $O(np)$ en vez de $O(n^2)$: nunca materializa la $n\times n$.

In [23]:
class EfficientQDA(TensorizedQDA):
    """P6) Esquiva la matriz n x n usando la identidad de P5:
    diag(A@B) = np.sum(A * B.T, axis=1). Equivale a sumar sobre las p features
    el producto elemento a elemento de unbiased con (inv_cov @ unbiased)."""

    def predict(self, X):
        unbiased = X[np.newaxis, :, :] - self.tensor_means            # (k, p, n)
        M = self.tensor_inv_cov @ unbiased                            # (k, p, n)
        quad = np.sum(unbiased * M, axis=1)                           # (k, n)
        log_dets = 0.5 * np.log(LA.det(self.tensor_inv_cov))[:, np.newaxis]
        log_conditionals = log_dets - 0.5 * quad
        log_posteriori = self.log_a_priori[:, np.newaxis] + log_conditionals
        return np.argmax(log_posteriori, axis=0).reshape(1, -1)

### Verificación: las 4 variantes de QDA predicen lo mismo

In [24]:
X_full, y_full = get_wine_dataset()
y_enc = label_encode(y_full)
Xtr, Xte, ytr, yte = split_transpose(X_full, y_enc, test_size=0.3, random_state=42)

qda_models = [QDA, TensorizedQDA, FasterQDA, EfficientQDA]
ref = None
print(f'{"model":16s} {"acc":>7s}  igual a QDA?')
for M in qda_models:
    m = M(); m.fit(Xtr, ytr); p = m.predict(Xte)
    if ref is None: ref = p
    acc = (yte.flatten() == p.flatten()).mean()
    print(f'{M.__name__:16s} {acc:7.4f}  {np.array_equal(p, ref)}')

model                acc  igual a QDA?
QDA               0.9815  True
TensorizedQDA     0.9815  True
FasterQDA         0.9815  True
EfficientQDA      0.9815  True


### P7 — Benchmark de las 4 variantes de QDA

In [25]:
cols = ['test_median_ms', 'test_speedup', 'test_mem_median_mb', 'test_mem_reduction', 'mean_accuracy']
b_wine = Benchmark(X_full, y_enc, n_runs=200, warmup=30, mem_runs=30, test_sz=0.3, same_splits=False)
for M in qda_models:
    b_wine.bench(M)
b_wine.summary(baseline='QDA')[cols].round(4)

Benching params:
Total runs: 260
Warmup runs: 30
Peak Memory usage runs: 30
Running time runs: 200
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/200 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/200 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/200 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/200 [00:00<?, ?it/s]

,test_median_ms,test_speedup,test_mem_median_mb,test_mem_reduction,mean_accuracy
model,,,,,
QDA,0.6515,1.0000,0.0077,1.0000,0.9826
TensorizedQDA,0.3082,2.1140,0.0121,0.6349,0.9851
FasterQDA,0.0182,35.6987,0.1092,0.0705,0.9848
EfficientQDA,0.0178,36.7042,0.0753,0.1023,0.9856


> **P7 · Comparar las 4 variantes de QDA. ¿Qué se observa? ¿Se condice con lo esperado?**

**Concepto.** La tabla revela *dónde estaba el cuello de botella*: no en la aritmética, sino en el overhead del intérprete de Python corriendo bucles.

**En lo técnico.** `TensorizedQDA` ~2× (solo saca el loop de clases). `FasterQDA`/`EfficientQDA` ~37× (sacan el loop de observaciones). `FasterQDA` usa más RAM que `EfficientQDA` por la matriz $n\times n$; en Wine ($n$ chico) la diferencia es leve y se amplifica en el dataset grande (P13).

**Respuesta.** El salto de velocidad viene de eliminar el `for` sobre observaciones, no de tensorizar las clases; y `EfficientQDA` logra ese mismo tiempo con memoria mínima. Se condice con lo esperado: *rápido* y *eficiente en memoria* son ejes distintos, y conviene apuntar a los dos.

## 3. Cholesky

> **Concepto de fondo.** Hasta acá optimizamos la predicción; queda el costo de invertir cada $\Sigma_j$ en el entrenamiento. Invertir una matriz general es $O(p^3)$ y numéricamente delicado; Cholesky factoriza $\Sigma=LL^T$ ($L$ triangular) y trabajar con la triangular es más barato y estable. (MML §4.3.)

---

> **P8 · Si $A=LL^T$, expresar $A^{-1}$ vía $L$. ¿Cómo ayuda en la forma cuadrática de QDA?**

**Concepto.** Nunca necesitamos $\Sigma^{-1}$ como objeto; solo la usamos dentro de la forma cuadrática. Cholesky la convierte en una norma al cuadrado —lo más barato y estable posible— y el log-determinante sale de la diagonal de $L$.

**En lo técnico.** $A^{-1}=(LL^T)^{-1}=L^{-T}L^{-1}=(L^{-1})^T(L^{-1})$. Con $y=L^{-1}(x-\mu)$:
$$(x-\mu)^T\Sigma^{-1}(x-\mu)=\lVert y\rVert^2,\qquad \tfrac12\log|\Sigma^{-1}|=\log\textstyle\prod_i(L^{-1})_{ii}.$$

**Respuesta.** $A^{-1}=(L^{-1})^T L^{-1}$; por eso la forma cuadrática es `(y**2).sum()` con $y=L^{-1}(x-\mu)$, y el determinante es el producto de la diagonal — todo sobre una triangular.

---

> **P9 · Diferencias entre `QDA_Chol1` y `QDA`, y cómo `QDA_Chol1` llega a la predicción.**

**Concepto.** Computan lo mismo por dos caminos: `QDA` el directo y caro (invierte $\Sigma$ entera, producto matricial completo); `QDA_Chol1` el atajo de P8 (factoriza, invierte solo la triangular, forma cuadrática = norma).

**En lo técnico.** `QDA_Chol1`: (1) $\Sigma_j=L_jL_j^T$; (2) guarda $L_j^{-1}$; (3) en `predict`: $y=L^{-1}\text{unbiased}$ → $\log(\prod\operatorname{diag}L^{-1})-\tfrac12\lVert y\rVert^2$.

**Respuesta.** `QDA_Chol1` cambia la inversión de $\Sigma$ y el producto matricial por una inversión triangular y una norma; llega al mismo número que `QDA` por la identidad de P8.

---

> **P10 · Diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`.**

**Concepto.** Las tres factorizan $\Sigma=LL^T$; difieren en una decisión de ingeniería: invertir $L$ de antemano vs resolver el sistema en cada predicción, y con qué rutina.

**En lo técnico.**

| | qué guarda | forma cuadrática | log-det |
|---|---|---|---|
| **Chol1** | $L^{-1}$ vía `LA.inv` (genérica) | $\lVert L^{-1}u\rVert^2$ | $+\log\prod\operatorname{diag}L^{-1}$ |
| **Chol2** | $L$ (no invierte) | `solve_triangular` $Ly=u$ | $-\log\prod\operatorname{diag}L$ |
| **Chol3** | $L^{-1}$ vía `dtrtri` (LAPACK triangular) | $\lVert L^{-1}u\rVert^2$ | $+\log\prod\operatorname{diag}L^{-1}$ |

**Respuesta.** Chol1 invierte $L$ con rutina genérica; Chol2 no invierte y resuelve el sistema por observación (log-det con signo opuesto, porque guarda $L$ y no $L^{-1}$); Chol3 invierte $L$ con `dtrtri`, especializada en triangulares.

### P11 — Benchmark de las 7 variantes (4 QDA + 3 Cholesky)

In [26]:
chol_models = [QDA_Chol1, QDA_Chol2, QDA_Chol3]
for M in chol_models:
    b_wine.bench(M)
order7 = ['QDA','TensorizedQDA','FasterQDA','EfficientQDA','QDA_Chol1','QDA_Chol2','QDA_Chol3']
b_wine.summary(baseline='QDA').loc[order7, cols].round(4)

QDA_Chol1 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol1 (TIME):   0%|          | 0/200 [00:00<?, ?it/s]

QDA_Chol2 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/200 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/200 [00:00<?, ?it/s]

,test_median_ms,test_speedup,test_mem_median_mb,test_mem_reduction,mean_accuracy
model,,,,,
QDA,0.6515,1.0000,0.0077,1.0000,0.9826
TensorizedQDA,0.3082,2.1140,0.0121,0.6349,0.9851
FasterQDA,0.0182,35.6987,0.1092,0.0705,0.9848
EfficientQDA,0.0178,36.7042,0.0753,0.1023,0.9856
QDA_Chol1,0.3696,1.7626,0.0078,0.9910,0.9831
QDA_Chol2,0.8611,0.7566,0.0080,0.9660,0.9814
QDA_Chol3,0.3701,1.7603,0.0076,1.0099,0.9822


> **P11 · Comparar las 7 variantes. ¿Hay alguna `QDA_Chol` claramente mejor o peor?**

**Concepto.** Entre las Cholesky, la diferencia es *dónde se pone el trabajo*: una vez en el `fit` (precomputar $L^{-1}$) o repetido en cada `predict` (resolver el sistema).

**En lo técnico.** Chol1 y Chol3 precomputan $L^{-1}$ → ~1.8×, parejas (difieren solo en la rutina del fit, `LA.inv` vs `dtrtri`). Chol2 resuelve `solve_triangular` por observación → ~0.78×, *por debajo* de `QDA`: el costo por-llamada no se amortiza.

**Respuesta.** Chol2 es claramente la peor en predicción; Chol1 y Chol3 son equivalentes y mejores. Para tensorizar conviene una que precompute $L^{-1}$ (elijo `QDA_Chol3` por el fit más barato), porque `solve_triangular` no batchea sobre el eje de clases.

> **P12 · Implementar `TensorizedChol`. P14 · Implementar `EfficientChol`.**

**Concepto.** Son las dos optimizaciones de la Sección 2 aplicadas a Cholesky: tensorizar sobre clases (`TensorizedChol`) y, además, eliminar el `for` y la matriz $n\times n$ (`EfficientChol`).

**En lo técnico.** Ambas heredan de `QDA_Chol3` y apilan $L^{-1}$ en `(k,p,p)`. `EfficientChol` calcula, para todas las observaciones, $y=L^{-1}\text{unbiased}$ `(k,p,n)` y $\lVert y\rVert^2=$ `np.sum(y*y, axis=1)` → `(k,n)`. (Código abajo.)

**Respuesta.** `TensorizedChol` paraleliza clases; `EfficientChol` combina sin-`for` + sin-$n\times n$ + Cholesky — la variante más completa.

In [27]:
class TensorizedChol(QDA_Chol3):
    """P12) Cholesky tensorizado: apila L^-1 en (k,p,p) y paraleliza sobre clases.
    Idea (P8): (x-mu)^T Sigma^-1 (x-mu) = ||L^-1 (x-mu)||^2 ; 0.5 log|Sigma^-1| = log prod diag(L^-1)."""

    def _fit_params(self, X, y):
        super()._fit_params(X, y)
        self.tensor_L_inv = np.stack(self.L_invs)                     # (k, p, p)
        self.tensor_means = np.stack(self.means)                     # (k, p, 1)
        self.log_dets = np.log(np.diagonal(self.tensor_L_inv, axis1=1, axis2=2)).sum(axis=1)

    def _predict_log_conditionals(self, x):
        unbiased = x - self.tensor_means                             # (k, p, 1)
        y = self.tensor_L_inv @ unbiased                            # (k, p, 1)
        quad = (y ** 2).sum(axis=(1, 2))                            # (k,)
        return self.log_dets - 0.5 * quad

    def _predict_one(self, x):
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))

In [28]:
class EfficientChol(QDA_Chol3):
    """P14) Combina EfficientQDA + TensorizedChol: sin for y sin matriz n x n.
    y = L^-1 @ unbiased (k,p,n); ||y||^2 por obs = np.sum(y*y, axis=1) -> (k,n)."""

    def _fit_params(self, X, y):
        super()._fit_params(X, y)
        self.tensor_L_inv = np.stack(self.L_invs)                     # (k, p, p)
        self.tensor_means = np.stack(self.means)                     # (k, p, 1)
        self.log_dets = np.log(np.diagonal(self.tensor_L_inv, axis1=1, axis2=2)).sum(axis=1)

    def predict(self, X):
        unbiased = X[np.newaxis, :, :] - self.tensor_means            # (k, p, n)
        y = self.tensor_L_inv @ unbiased                             # (k, p, n)
        quad = np.sum(y * y, axis=1)                                 # (k, n)
        log_conditionals = self.log_dets[:, np.newaxis] - 0.5 * quad
        log_posteriori = self.log_a_priori[:, np.newaxis] + log_conditionals
        return np.argmax(log_posteriori, axis=0).reshape(1, -1)

### Verificación: las 9 variantes predicen idéntico

In [29]:
all_models = [QDA, TensorizedQDA, FasterQDA, EfficientQDA,
              QDA_Chol1, QDA_Chol2, QDA_Chol3, TensorizedChol, EfficientChol]
ref = None; ok = True
for M in all_models:
    m = M(); m.fit(Xtr, ytr); p = m.predict(Xte)
    if ref is None: ref = p
    same = np.array_equal(p, ref); ok &= same
    print(f'{M.__name__:16s} igual a QDA? {same}')
print('\nTODAS idénticas:', ok)

QDA              igual a QDA? True
TensorizedQDA    igual a QDA? True
FasterQDA        igual a QDA? True
EfficientQDA     igual a QDA? True
QDA_Chol1        igual a QDA? True
QDA_Chol2        igual a QDA? True
QDA_Chol3        igual a QDA? True
TensorizedChol   igual a QDA? True
EfficientChol    igual a QDA? True

TODAS idénticas: True


### P13 — Benchmark de las 9 variantes (Wine)

In [30]:
for M in [TensorizedChol, EfficientChol]:
    b_wine.bench(M)
order9 = order7 + ['TensorizedChol', 'EfficientChol']
b_wine.summary(baseline='QDA').loc[order9, cols].round(4)

TensorizedChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedChol (TIME):   0%|          | 0/200 [00:00<?, ?it/s]

EfficientChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

EfficientChol (TIME):   0%|          | 0/200 [00:00<?, ?it/s]

,test_median_ms,test_speedup,test_mem_median_mb,test_mem_reduction,mean_accuracy
model,,,,,
QDA,0.6515,1.0000,0.0077,1.0000,0.9826
TensorizedQDA,0.3082,2.1140,0.0121,0.6349,0.9851
FasterQDA,0.0182,35.6987,0.1092,0.0705,0.9848
EfficientQDA,0.0178,36.7042,0.0753,0.1023,0.9856
QDA_Chol1,0.3696,1.7626,0.0078,0.9910,0.9831
QDA_Chol2,0.8611,0.7566,0.0080,0.9660,0.9814
QDA_Chol3,0.3701,1.7603,0.0076,1.0099,0.9822
TensorizedChol,0.1517,4.2938,0.0124,0.6189,0.9826
EfficientChol,0.0122,53.5465,0.0609,0.1264,0.9836


### P13 (cont.) — Dataset grande: la asíntota de memoria

En Wine ($n$ chico, $k=3$) las diferencias de memoria casi no se ven. Repito el
benchmark en un subsample de **letters** ($k=26$, $p=16$), donde la matriz
$n\times n$ de `FasterQDA` pesa de verdad: con $n_{\text{test}}\approx 1000$ y $k=26$
son $(26,1000,1000)$ floats $\approx 208$ MB.

In [31]:
from numpy.random import RandomState
Xl, yl = get_letters_dataset(); yl = label_encode(yl.reshape(-1,1))
rs = RandomState(0); idx = rs.choice(len(Xl), 4000, replace=False)
Xl, yl = Xl[idx], yl[idx]
print('letters subsample:', Xl.shape, 'k =', len(np.unique(yl)))

b_let = Benchmark(Xl, yl, n_runs=25, warmup=5, mem_runs=10, test_sz=0.25, same_splits=False)
for M in all_models:
    b_let.bench(M)
b_let.summary(baseline='QDA').loc[order9, cols].round(4)

letters subsample: (4000, 16) k = 26
Benching params:
Total runs: 40
Warmup runs: 5
Peak Memory usage runs: 10
Running time runs: 25
Train size rows (approx): 3000
Test size rows (approx): 1000
Test size fraction: 0.25


QDA (MEM):   0%|          | 0/10 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/25 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/10 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/25 [00:00<?, ?it/s]

...

TensorizedChol (MEM):   0%|          | 0/10 [00:00<?, ?it/s]

TensorizedChol (TIME):   0%|          | 0/25 [00:00<?, ?it/s]

EfficientChol (MEM):   0%|          | 0/10 [00:00<?, ?it/s]

EfficientChol (TIME):   0%|          | 0/25 [00:00<?, ?it/s]

,test_median_ms,test_speedup,test_mem_median_mb,test_mem_reduction,mean_accuracy
model,,,,,
QDA,102.9034,1.0000,0.0746,1.0000,0.8642
TensorizedQDA,21.5611,4.7726,0.1312,0.5686,0.8622
FasterQDA,2.7462,37.4711,204.8284,0.0004,0.8614
EfficientQDA,0.6509,158.1001,9.8368,0.0076,0.8595
QDA_Chol1,54.0739,1.9030,0.0723,1.0321,0.8618
QDA_Chol2,133.1170,0.7730,0.0722,1.0327,0.8606
QDA_Chol3,53.9280,1.9082,0.0720,1.0369,0.8635
TensorizedChol,3.4619,29.7244,0.1349,0.5531,0.8620
EfficientChol,0.4621,222.6947,9.8371,0.0076,0.8616


> **P13 · Comparar las 9 variantes. ¿Qué se observa? ¿Se condice con lo esperado?**

**Concepto.** Wine era demasiado chico para distinguir los dos ejes que importan; el dataset grande los separa: **tiempo** y **memoria**.

**En lo técnico.** En tiempo dominan las *Efficient* (`EfficientChol` ~210×, `EfficientQDA` ~160×). En memoria, `FasterQDA` explota a ~205 MB (la $n\times n$) frente a ~10 MB de las *Efficient*. `QDA_Chol2` vuelve a ser la peor en predict; las *Tensorized* quedan en el medio (paralelizan clases, no observaciones).

**Respuesta.** El mejor modelo combina las tres ideas —sin `for`, sin $n\times n$, con Cholesky— y es `EfficientChol`. Se condice con lo esperado, más nítido que en Wine: optimizar fue apilar mejoras sin tocar la corrección (las 9 predicen idéntico).

## 4. Accuracy por Cross-Validation (5-fold estratificado)

El `Benchmark` reporta `mean_accuracy` sobre repeated train/test splits aleatorios:
sirve para medir tiempo/memoria, pero como **métrica de calidad** es ruidosa (y con
`same_splits=False` cada modelo ve splits distintos). Para una estimación robusta uso
`StratifiedKFold`: cada observación se evalúa exactamente una vez como test y las
clases quedan balanceadas en cada fold. Como las 9 variantes predicen idéntico, la
accuracy de CV es la misma para todas — reporto el modelo y, de paso, confirmo que la
implementación no degrada la calidad.

In [32]:
from sklearn.model_selection import StratifiedKFold

def cv_accuracy(model_class, X, y, n_splits=5, seed=6553, **kwargs):
    """Accuracy por k-fold estratificado. X:(N,p), y:(N,1). Los modelos esperan (p,n)."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    y_flat = y.flatten(); accs = []
    for tr, te in skf.split(X, y_flat):
        m = model_class(**kwargs); m.fit(X[tr].T, y[tr].T)
        preds = m.predict(X[te].T)
        accs.append((y_flat[te] == preds.flatten()).mean())
    return np.array(accs)

for name, M in [('QDA', QDA), ('EfficientQDA', EfficientQDA), ('EfficientChol', EfficientChol)]:
    accs = cv_accuracy(M, X_full, y_enc, n_splits=5)
    print(f'{name:16s} CV acc = {accs.mean():.4f} +/- {accs.std():.4f}   folds={np.round(accs,3)}')

QDA              CV acc = 0.9887 +/- 0.0138   folds=[1.    0.972 1.    1.    0.971]
EfficientQDA     CV acc = 0.9887 +/- 0.0138   folds=[1.    0.972 1.    1.    0.971]
EfficientChol    CV acc = 0.9887 +/- 0.0138   folds=[1.    0.972 1.    1.    0.971]


## 5. Conclusiones

Más allá de los números, lo que deja el TP son algunas ideas que trascienden a QDA:

1. **Optimizar es, ante todo, encontrar dónde se va el tiempo — no calcular más rápido.** El gran salto no vino de aritmética más astuta sino de sacar los bucles de Python: el intérprete era el costo dominante, no las operaciones. Medir antes de optimizar es lo que reveló esto.

2. **Vectorizar "de una" puede esconder un costo cuadrático.** `FasterQDA` es la trampa elegante: corre rápido pero construye una matriz $n\times n$ de la que solo usa la diagonal. La lección es que *rápido* y *eficiente* son ejes distintos, y mirar solo el reloj te puede hacer elegir un modelo que no escala.

3. **Un poco de álgebra ahorra mucho cómputo.** La identidad $\operatorname{diag}(AB)=\sum_{\text{cols}}A\odot B^T$ (P5) y la factorización de Cholesky (P8) no son trucos de NumPy: son resultados matemáticos que bajan la complejidad de $O(n^2)$ a $O(np)$ y reemplazan una inversión general por una norma. Pensar la matemática *antes* de codear fue lo que habilitó las mejores variantes.

4. **Las optimizaciones se componen.** El mejor modelo, `EfficientChol`, no aplica una idea sino tres apiladas (sin `for` + sin $n\times n$ + Cholesky). Ninguna anula a las otras; se suman.

5. **Optimizar no es cambiar el modelo.** Las 9 variantes predicen *exactamente* lo mismo (verificado byte a byte) y la accuracy por cross-validation lo confirma. Toda la ganancia fue en *cómo* se computa, manteniendo intacto *qué* se computa — que es la única forma honesta de optimizar.